# Fiber flux for split exposures (`telemetry_mining` rewrite)

Rewrite of `linphi_splitflux.ipynb` using the `telemetry_mining` package. Kept deliberately
verbose (extra markdown + inline comments) as a worked example of what `Exposure(expid)`
buys you, since this notebook happens to touch three quite different sources through the
*same* object: the condensed per-exposure DB record (`db_row`), the fiber coordinates file
(`coords`), and per-camera spectral-reduction tables (`cframe_table`). The original notebook
(manual `fitsio` opens + `DOSlib.util.coords2df`) is left untouched for comparison; see
`API.md` for the full accessor reference and `FIELDS.md` for a column-by-column glossary of
everything these accessors expose.

**The science question**: exposures 255008/255009/255010 are a rapid sequence of *split*
exposures of the same field on the same night (three short back-to-back exposures instead
of one longer one). If the fiber positioners and flux calibration are stable, the same star
observed in each of the three exposures should show consistent flux. Any systematic
difference between exposures -- especially one that correlates with petal, or with whether
a positioner has the "linphi" issue -- is a QA signal worth chasing.

- The data comes from the `cframe` files at NERSC (one file per camera per exposure, the
  per-fiber output of the spectral extraction pipeline).
- The `FIBERMAP` extension (targeting/positioning info) is used to select stars via
  `MORPHTYPE == 'PSF'` (point sources -- excludes resolved galaxies) and
  `GAIA_PHOT_G_MEAN_MAG < 18` (bright enough to measure well, but not so bright they're
  used as spectrophotometric standards themselves).
- The `SCORES` extension carries the actual fiber flux measurements
  (`MEDIAN_CALIB_COUNT`/`MEDIAN_CALIB_SNR` -- post-flux-calibration counts and S/N per
  fiber; the two give qualitatively identical results for this comparison).
- `POS_LINPHI` -- whether a positioner's phi arm is in the "linear" calibration regime, i.e.
  whether it has the linphi issue this notebook is ultimately named for -- comes from the
  *coordinates* file, a different source than `FIBERMAP`/`SCORES`. It's joined in below so
  it's available for a future linphi-vs-regular-positioner split; not yet used in the plots
  below, same as the original.
- Runs over multiple exposures/cameras, split by petal. Multiple obsdays is prepared but not
  fully implemented, same caveat as the original.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Add telemetry_mining to the path. Point DOS_TELEMETRY_MINING_DIR at your own
# checkout (svn co https://desi.lbl.gov/svn/code/online/telemetry_mining/trunk);
# defaults to the maintainer's checkout at NERSC.
TM_DIR = os.getenv("DOS_TELEMETRY_MINING_DIR", os.path.expanduser("~/telemetry_mining-trunk/src"))
sys.path.insert(0, TM_DIR)
from telemetry_mining import Exposure

## Setup

One field, three consecutive short ("split") exposures, all 30 cameras (10 petals x
`{b,r,z}` spectrograph arms -- DESI's spectrographs cover the blue/red/near-infrared in
three separate arms per petal, hence 3x10 = 30 files per exposure).

In [ ]:
obsdays = [20240925]       # for now only single obsdays are supported
expids = [255008, 255009, 255010]
cameras = ['z%1d' % x for x in range(10)]
cameras += ['r%1d' % x for x in range(10)]
cameras += ['b%1d' % x for x in range(10)]

# you can overwrite cameras if you want to use only a subset like this
# cameras = ['z0']

print(f' These cameras {cameras} are used with exposures {expids}')

## The `Exposure` accessor

`Exposure(expid, night=...)` is a lazy, cached handle onto *everything* associated with one
exposure ID -- constructing it does zero I/O. Nothing is read from disk or queried from the
database until you actually touch an attribute (`.header`, `.db_row`, `.coords`,
`.cframe_table(camera)`, ...), and each attribute's result is cached on the instance after
the first access, so touching it again later is free. Passing `night=` explicitly here
(rather than leaving it to be resolved automatically) skips one DB round-trip per exposure,
since we already know it.

Building one dict of `Exposure` objects up front and reusing those same instances
everywhere below (rather than constructing a fresh `Exposure(exp)` inside every loop) is
what makes that caching actually pay off -- each exposure's DB record, coordinates file, and
per-camera cframe tables each get read/queried at most once for this whole notebook.

In [ ]:
exposures = {exp: Exposure(exp, night=obsdays[0]) for exp in expids}

In [ ]:
# A quick look at what's available -- the telescope control system state for one exposure
exposures[expids[1]].db_row['tcs']

`db_row` is the condensed, one-row-per-exposure record from the `exposure.exposure`
database table -- 187 columns, several of them (like `tcs` above) whole JSON blobs rather
than scalars. `tcs` is the telescope control system's state snapshot for this exposure
(pointing, tracking status, guider offsets, ...); other exposures carry similar blobs for
`etc`/`guider`/`astrometry`/`hexapod` and more. This cell is purely exploratory -- nothing
below depends on it -- included to show what's available if you need it. See `FIELDS.md`
for the full column list (including what's inside each jsonb block) with real example
values.

## cframe tables + linphi flag

This is the heart of what `Exposure` buys you here. The original notebook, per camera, per
exposure: built a file path by hand, opened it with `fitsio`, read the `FIBERMAP` and
`SCORES` extensions separately, fixed their FITS big-endian byte order (`.byteswap()
.newbyteorder()` -- broken by NumPy 2.0's removal of `ndarray.newbyteorder()`, silently, so
this was also a latent bug in the original), wrapped each in a `pandas.DataFrame`, and
concatenated them side by side. `Exposure.cframe_table(camera)` does all of that in one
call, already indexed by `(PETAL_LOC, DEVICE_LOC)` -- one row per fiber. A single camera
(e.g. `'z3'`) is one spectrograph arm on one petal, so that's 500 rows -- the fibers
belonging to that one petal -- with `FIBERMAP`'s targeting columns and `SCORES`'s flux
columns already combined.

That `(PETAL_LOC, DEVICE_LOC)` index is what makes the `POS_LINPHI` join below trivial:
`Exposure.coords` reads the separate `coordinates-<expid>.fits` file (fiber positions from
the positioner control system, not the spectral pipeline) and uses the *same* indexing
convention, so joining "does this fiber's positioner have the linphi issue" onto "what flux
did this fiber measure" is a plain `pandas.concat(..., axis=1)` rather than any kind of
manual matching.

In [ ]:
frames = {}
printed = False
for day in obsdays:
    print(f'Processing obsday {day}')
    frames[day] = {}
    for exp in expids:
        print(f'Processing exposure {exp}')
        e = exposures[exp]
        # e.coords is read once per exposure (cached on `e`), indexed by (PETAL_LOC, DEVICE_LOC).
        linphi = e.coords[['POS_LINPHI']]
        frames[day][exp] = {}
        # cframe_tables reads every camera's FIBERMAP+SCORES in one process-parallel batch (each
        # camera is a separate gzip file), instead of one blocking read per camera in a loop.
        tables, errors = e.cframe_tables(cameras)
        for camera, combined in tables.items():
            # inner join on (PETAL_LOC, DEVICE_LOC): keeps fibers present in both this camera's
            # 500-fiber table and the coordinates file (should be all of them).
            frames[day][exp][camera] = pd.concat([combined, linphi], join='inner', axis=1)
        if errors:
            print(f'  {len(errors)} camera(s) unavailable: {sorted(errors)}')
        if not printed and frames[day][exp]:
            printed = True
            cam0 = next(iter(frames[day][exp]))
            print(f'Available columns: {frames[day][exp][cam0].columns.tolist()}')
print('Done')


## Select stars

Filter each camera's combined table down to point sources bright enough to measure well:
`MORPHTYPE == 'PSF'` keeps point sources (stars, quasars) and excludes resolved galaxies
(whose measured flux depends on fiber placement relative to an extended profile, which
would confound a flux-*repeatability* test); `GAIA_PHOT_G_MEAN_MAG < 18` keeps only
reasonably bright ones, so photon noise doesn't dominate the exposure-to-exposure
comparison. Same cut as the original.

In [ ]:
_stars = {}
for day in obsdays:
    _stars[day] = {}
    for exp in expids:
        _stars[day][exp] = {}
        for camera in cameras:
            df = frames[day][exp][camera]
            morph = df[df['MORPHTYPE'] == 'PSF']
            m = morph[morph['GAIA_PHOT_G_MEAN_MAG'] < 18.]
            # a second cut (for testing)
            # stars[day][exp][camera] = m[m['GAIA_PHOT_G_MEAN_MAG'] > 10.0]
            _stars[day][exp][camera] = m

In [ ]:
linphi_stars = {}
regular_stars = {}
for day in _stars:
    linphi_stars[day] = {}
    regular_stars[day] = {}
    for exp in _stars[day]:
        linphi_stars[day][exp] = {}
        regular_stars[day][exp] = {}
        for camera in _stars[day][exp]:
            df = _stars[day][exp][camera]
            linphi_stars[day][exp][camera] = df[df['POS_LINPHI'] == 'True']
            regular_stars[day][exp][camera] = df[df['POS_LINPHI'] == 'False']

## Histograms and per-exposure/per-petal comparisons

Unchanged from the original -- same plots, operating on `stars` regardless of how its
columns were computed above (this section doesn't touch `Exposure` at all any more; the
work is already done). `MEDIAN_CALIB_COUNT`/`MEDIAN_CALIB_SNR` are per-fiber, post-flux
calibration counts/S/N from `SCORES`. First the raw distributions per exposure, then
exposure-to-exposure *differences* for the same star (overall, then split out per petal,
since a systematic per-petal offset -- rather than just scatter -- is the kind of thing that
would point at a positioner or calibration issue specific to that petal).

In [ ]:
# Select which stars (underlying robot type) to use
stars = [regular_stars, linphi_stars]
star_text = ["regular Robots", "Linphi Robots"]

In [ ]:
# Histogram fiber flux (variable name is hardcoded)
plt.figure(figsize=(10,5))
for i in range(2):
    plt.subplot(1,2,i+1)
    for day in stars[i].keys():
        for exp in stars[i][day].keys():
            x = []
            for camera in stars[i][day][exp].keys():
                C = camera[0].upper()
                x += list(stars[i][day][exp][camera][f'MEDIAN_CALIB_COUNT_{C}'])
            plt.hist(x,bins=100,range=(0,40))
            plt.legend(stars[i][day].keys())
            plt.title(f'MEDIAN_CALIB_COUNT ({star_text[i]})')
plt.show()

In [ ]:
# Histogram fiberflux
plt.figure(figsize=(10,5))
for i in range(2):
    plt.subplot(1,2,i+1)
    for day in stars[i].keys():
        for exp in stars[i][day].keys():
            x = []
            for camera in stars[i][day][exp].keys():
                C = camera[0].upper()
                x += list(stars[i][day][exp][camera][f'MEDIAN_CALIB_SNR_{C}'])
            plt.hist(x,bins=50,range=(0,4))
            plt.legend(stars[i][day].keys())
        plt.title(f'MEDIAN_CALIB_SNR ({star_text[i]})')
plt.show()

In [ ]:
# Histogram fiberflux difference
field = 'MEDIAN_CALIB_COUNT' #'MEDIAN_CALIB_SNR'
plt.figure(figsize=(10,5))
for i in range(2):
    for day in stars[i].keys():
        plt.subplot(1,2,i+1)
        expids_ = list(stars[i][day].keys())
        first = expids_.pop(0)
        for exp in expids_:
            delta = []
            for camera in stars[i][day][exp].keys():
                C = camera[0].upper()
                x1=list(stars[i][day][first][camera][field+'_'+C])
                x2=list(stars[i][day][exp][camera][field+'_'+C])
                for j in range(len(x1)):
                    delta.append(x1[j]-x2[j])
            plt.hist(delta,bins=50,range=(-0.2,.2),label=f'{first}-{exp}') 
            first = exp
        #add difference between very first and last exposure if len(expids_)>2
        expids_ = list(stars[i][day].keys())
        if len(expids_)>2:
            first = expids_[0]
            exp = expids_[-1]
            delta=[]
            for camera in stars[i][day][exp].keys():
                C = camera[0].upper()
                x1=list(stars[i][day][first][camera][field+'_'+C])
                x2=list(stars[i][day][exp][camera][field+'_'+C])
                for j in range(len(x1)):
                    delta.append(x1[j]-x2[j])
            plt.hist(delta,bins=50,range=(-0.2,.2),histtype='step',color='r',label=f'{first}-{exp}')    
        plt.legend()
        plt.title(f'{field} difference ({star_text[i]})')
plt.show()

In [ ]:
# Histogram fiberflux difference per petal
field = 'MEDIAN_CALIB_COUNT'
plt.figure(figsize=(10,15))
legend = []
deltas = []
for s in range(2):
    row=-1
    deltas.append({})
    for day in stars[s].keys():
        expids_ = list(stars[s][day].keys())
        pairs = []
        l = len(expids_)
        for i in range(l):
            if len(expids_)<2:
                break
            first = expids_.pop(0)
            for exp in expids_:
                pairs.append((first,exp))
    
        for first, exp in pairs:
            row = row+1
            delta = {}
            for camera in stars[s][day][exp].keys():
                C = camera[0].upper()
                P = camera[1].upper()
                if P not in delta:
                    delta[P] = []
                if f'Petal {P}' not in legend:
                    legend.append(f'Petal {P}')
                x1=list(stars[s][day][first][camera][field+'_'+C])
                x2=list(stars[s][day][exp][camera][field+'_'+C])
                for i in range(len(x1)):
                    delta[P].append(x1[i]-x2[i])
            t = f'({first} - {exp})'
            deltas[s][t] = delta
            plt.subplot(3,2,s+2*row+1)
            for p in delta.keys():
                plt.hist(delta[p],bins=50,range=(-0.2,.2),histtype='bar',alpha=0.5, label=f'Petal {p}')    
            plt.legend()
            plt.title(f'{field} {t} ({star_text[s]})',fontsize=10)
plt.show()

In [ ]:
# plot mean
means = {}
all_means = {}
plt.figure(figsize=(10,5))
for s in range(2):
    plt.subplot(1,2,s+1)
    for i in deltas[s].keys():
        means[i] = []
        all_petals = []
        for p in deltas[s][i].keys():
            x=np.array(deltas[s][i][p])
            x=x[x<0.5]
            x=x[x>-0.5]
            all_petals += list(x)
            means[i].append(x.mean())
        all_means[i] = np.array(all_petals).mean()
    ax = plt.gca()
    ax.set_ylim([-0.2,0.2])
    for i in means.keys():
        plt.plot(np.linspace(0,10,num=10),means[i],marker='*',linestyle='-',label=i)
    plt.title(f'Mean diff ({field}) per petal ({star_text[s]})',fontsize=8)
    plt.legend()
    print(f'Mean diff ({field}) all petals')
    for i in means.keys():
        print(f'\tBetween exposures {i}: {round(all_means[i],4)} ({star_text[s]})')
plt.show()


In [ ]:
# Histogram fiberflux difference per petal (regular robots)
field = 'MEDIAN_CALIB_COUNT'
_stars = stars[0]
st = star_text[0]
for day in _stars.keys():
    expids_ = list(_stars[day].keys())
    first = expids_.pop(0)
    legend = []
    deltas_list = []
    for exp in expids_:
        delta = {}
        for camera in _stars[day][exp].keys():
            C = camera[0].upper()
            P = camera[1].upper()
            if P not in delta:
                delta[P] = []
            if f'Petal {P}' not in legend:
                legend.append(f'Petal {P}')
            x1=list(_stars[day][first][camera][field+'_'+C])
            x2=list(_stars[day][exp][camera][field+'_'+C])
            for i in range(len(x1)):
                delta[P].append(x1[i]-x2[i])
        if f'Petal {P}' not in legend:
            legend.append(f'Petal {P}')
        deltas_list.append(delta)
        fig = plt.figure(figsize=(10,10))
        for p in delta.keys():
            fig.suptitle(f'{field} difference per petal for {first} - {exp} ({st})')
            plt.subplot(5,2,int(p)+1)
            n,_,_ = plt.hist(delta[p],bins=50,range=(-0.2,0.2),histtype='bar',alpha=0.5) 
            plt.legend([f'Petal {p}'])
            # add mean
            x=np.array(delta[p])
            x = x[x<0.5]
            x = x[x>-0.5]
            mean=x.mean()
            plt.vlines(x=mean, ymin=0.0,ymax=max(n),color='r')          
        first = exp
        plt.show()

In [ ]:
# Histogram fiberflux difference per petal (lin phi robots)
field = 'MEDIAN_CALIB_COUNT'
_stars = stars[1]
st = star_text[1]
for day in _stars.keys():
    expids_ = list(_stars[day].keys())
    first = expids_.pop(0)
    legend = []
    deltas_list = []
    for exp in expids_:
        delta = {}
        for camera in _stars[day][exp].keys():
            C = camera[0].upper()
            P = camera[1].upper()
            if P not in delta:
                delta[P] = []
            if f'Petal {P}' not in legend:
                legend.append(f'Petal {P}')
            x1=list(_stars[day][first][camera][field+'_'+C])
            x2=list(_stars[day][exp][camera][field+'_'+C])
            for i in range(len(x1)):
                delta[P].append(x1[i]-x2[i])
        if f'Petal {P}' not in legend:
            legend.append(f'Petal {P}')
        deltas_list.append(delta)
        fig = plt.figure(figsize=(10,10))
        for p in delta.keys():
            fig.suptitle(f'{field} difference per petal for {first} - {exp} ({st})')
            plt.subplot(5,2,int(p)+1)
            n,_,_ = plt.hist(delta[p],bins=50,range=(-0.2,0.2),histtype='bar',alpha=0.5) 
            plt.legend([f'Petal {p}'])
            # add mean
            x=np.array(delta[p])
            x = x[x<0.5]
            x = x[x>-0.5]
            mean=x.mean()
            plt.vlines(x=mean, ymin=0.0,ymax=max(n),color='r')          
        first = exp
        plt.show()

## Recap

What `Exposure` replaced here, concretely:

| Original (manual) | This notebook |
|---|---|
| Build `cframe-<camera>-<expid>.fits.gz` path by hand | `e.cframe_table(camera)` |
| `fitsio.FITS(...)`, read `FIBERMAP`/`SCORES`, fix byte order, concat | (same call) |
| `DOSlib.util.coords2df(exp)` (a personal, unpackaged checkout with a known bug) | `e.coords` |
| Re-open/re-query per use if referenced more than once | Cached on `e` after first access |
